# Qualidade dos Dados e Tratamento

Este notebook realiza o processo de análise voltado à qualidade dos dados, com o objetivo de gerar alguns tratamentos prévios sobre a base simulada do projeto ***Turnover Crítico***.

Assim sendo, o objetivo nesta etapa é:

- Validar a integridade dos dados (estrutura, linhas, tipos)
- Detectar valores nulos, incoerentes ou extremos (outliers)
- Garantir consistência entre as tabelas (temporalidade dos dados)
- Criar variáveis derivadas importantes para análises e hipóteses
- Preparar a base final para EDA e modelagem

## Imports e carregamento dos dados

In [1]:
# ----------------------------
# Configuração inicial e leitura das bases
# ----------------------------

import time
start = time.perf_counter()

import pandas as pd
import numpy as np
from datetime import date

# Carregar datasets simulados a partir da pasta raw
colab = pd.read_csv("../data/raw/dim_colaboradores.csv")
rh_mensal = pd.read_csv("../data/raw/fato_rh_mensal.csv")
promocoes = pd.read_csv("../data/raw/eventos_promocoes.csv")
desligamentos = pd.read_csv("../data/raw/eventos_desligamentos.csv")

## Análise de Qualidade (Data Quality)

### Validar carregamento

In [2]:
# ----------------------------
# Print dos dados
# ----------------------------

display(colab.head(1))
display(rh_mensal.head(1))
display(promocoes.head(1))
display(desligamentos.head(1))

,colaborador_id,area,nivel_cargo,genero,raca,data_admissao,idade,salario_base_inicial
0,1,Tecnologia,Sênior,F,Parda,2018-09-25,37,11366.628604


,colaborador_id,mes_ano,engajamento,performance,horas_extras
0,1,2024-01-01,66.406476,2.336944,12.857885


,colaborador_id,data_promocao,nivel_anterior,nivel_novo
0,1,2024-05-01,Sênior,Líder


,colaborador_id,data_desligamento,tipo_desligamento,custo_reposicao
0,42,2024-02-01,Involuntário,35928.607687


### Estrutura das tabelas e tipo de dados

In [3]:
# ----------------------------
# Schema das tabelas
# ----------------------------

print(f"Tabela dim_colaboradores - Linhas: {colab.shape[0]} e Colunas: {colab.shape[1]}")
print(f"Tabela fato_rh_mensal:  - Linhas: {rh_mensal.shape[0]} e Colunas: {rh_mensal.shape[1]}")
print(f"Tabela eventos_promocoes:  - Linhas: {promocoes.shape[0]} e Colunas: {promocoes.shape[1]}")
print(f"Tabela eventos_desligamentos:  - Linhas: {desligamentos.shape[0]} e Colunas: {desligamentos.shape[1]}")

Tabela dim_colaboradores - Linhas: 2500 e Colunas: 8
Tabela fato_rh_mensal:  - Linhas: 60000 e Colunas: 5
Tabela eventos_promocoes:  - Linhas: 350 e Colunas: 4
Tabela eventos_desligamentos:  - Linhas: 130 e Colunas: 4


In [4]:
# ----------------------------
# Informações gerais
# ----------------------------

print("Informações - Tabela de Colaboradores\n")
colab.info()

print("\nInformações - Tabela de RH Mensal\n")
rh_mensal.info()

print("\nInformações - Tabela de Promoções\n")
promocoes.info()

print("\nInformações - Tabela de Desligamentos\n")
desligamentos.info()

Informações - Tabela de Colaboradores

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   colaborador_id        2500 non-null   int64  
 1   area                  2500 non-null   str    
 2   nivel_cargo           2500 non-null   str    
 3   genero                2500 non-null   str    
 4   raca                  2500 non-null   str    
 5   data_admissao         2500 non-null   str    
 6   idade                 2500 non-null   int64  
 7   salario_base_inicial  2500 non-null   float64
dtypes: float64(1), int64(2), str(5)
memory usage: 156.4 KB

Informações - Tabela de RH Mensal

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   colaborador_id  60000 non-null  int64  
 1   mes_ano         60000 n

### Dados ausentes/nulos

- Na leitura das informações acima, já temos a informação de que não há valores nulos, no entanto, preferi realizar uma nova verificação pelo método isna()

In [5]:
# ----------------------------
# Nulos
# ----------------------------

print("Nulos por tabela:")
print("\nDim Colaboradores:")
print(colab.isna().sum())

print("\nFato RH Mensal:")
print(rh_mensal.isna().sum())

print("\nPromoções:")
print(promocoes.isna().sum())

print("\nDesligamentos:")
print(desligamentos.isna().sum())

Nulos por tabela:

Dim Colaboradores:
colaborador_id          0
area                    0
nivel_cargo             0
genero                  0
raca                    0
data_admissao           0
idade                   0
salario_base_inicial    0
dtype: int64

Fato RH Mensal:
colaborador_id    0
mes_ano           0
engajamento       0
performance       0
horas_extras      0
dtype: int64

Promoções:
colaborador_id    0
data_promocao     0
nivel_anterior    0
nivel_novo        0
dtype: int64

Desligamentos:
colaborador_id       0
data_desligamento    0
tipo_desligamento    0
custo_reposicao      0
dtype: int64


### Tratamento de Dados (tipos e datas)

- As colunas de datas das tabelas de Colaboradores, RH Mensal (indicadores de performance, engajamento e horas extras), Promoções e Desligamentos, estão com as suas datas como tipo "string", então vou alterar para formato de data

In [6]:
# ----------------------------
# Conversão de tipos de dados
# ----------------------------

colab["data_admissao"] = pd.to_datetime(colab["data_admissao"])
rh_mensal["mes_ano"] = pd.to_datetime(rh_mensal["mes_ano"])
promocoes["data_promocao"] = pd.to_datetime(promocoes["data_promocao"])
desligamentos["data_desligamento"] = pd.to_datetime(desligamentos["data_desligamento"])

### Checar a consistência de tempo dos dados

- Por exemplo, o desligamento não pode ser antes da demissão, se houver isso, os dados estão com algum tipo de erro/inconsistência

In [7]:
# ----------------------------
# Análise de consistência de datas
# ----------------------------

erros_adm = desligamentos[
    desligamentos["data_desligamento"] < colab.set_index("colaborador_id").loc[
        desligamentos["colaborador_id"], "data_admissao"
    ].values
]

print("Desligamentos com datas inválidas:", erros_adm.shape[0])

Desligamentos com datas inválidas: 9


A célula acima informa que temos 9 desligamentos com datas inválidas, nesse caso, pensando em um cenário de dados reais, poderia ter sido algum erro de cadastro/digitação ou mesmo por erro de integração de fontes distintas, normalização de datas e/ou erros em termos de datas. 

- ***Neste caso, como é um projeto fictício e a ocorrência tem baixa frequência, optarei pela exclusão desses 9 casos.***

In [8]:
# ----------------------------
# Exclusão dos erros nas datas de desligamentos
# ----------------------------

desli = desligamentos.merge(
    colab[["colaborador_id", "data_admissao"]],
    on="colaborador_id",
    how="left",
    validate="m:1" # inclui esse parãmetro apenas pela questão de relacionamento muitos para um, para evitar erro de duplicidade
)

desli["flag_inconsistencia"] = desli["data_desligamento"] < desli["data_admissao"]
qtd_invalidos = desli["flag_inconsistencia"].sum()
print(f"Desligamentos inválidos detectados: {qtd_invalidos}")

if qtd_invalidos > 0:
    display(desli.loc[desli["flag_inconsistencia"], 
                      ["colaborador_id", "data_admissao", "data_desligamento", "tipo_desligamento"]].head(10))

desli_corrigido = desli.loc[~desli["flag_inconsistencia"]].copy()

qtd_invalidos_pos = (desli_corrigido["data_desligamento"] < desli_corrigido["data_admissao"]).sum()
print(f"Inválidos após tratamento (esperado=0): {qtd_invalidos_pos}")

# 8) Atualizar objeto 'desligamentos' com a versão corrigida (sem colunas auxiliares)
desligamentos = desli_corrigido[
    ["colaborador_id", "data_desligamento", "tipo_desligamento", "custo_reposicao"]
].reset_index(drop=True)

print(f"Formato final de 'desligamentos' após limpeza: {desligamentos.shape}")

Desligamentos inválidos detectados: 9


,colaborador_id,data_admissao,data_desligamento,tipo_desligamento
3,131,2024-12-05,2024-10-01,Voluntário
8,223,2024-08-23,2024-01-01,Voluntário
49,1058,2024-05-13,2024-05-01,Voluntário
62,1287,2024-06-19,2024-01-01,Involuntário
63,1320,2024-06-18,2024-01-01,Voluntário
101,2103,2024-10-14,2024-10-01,Voluntário
111,2214,2024-10-28,2024-07-01,Involuntário
116,2293,2024-11-16,2024-05-01,Involuntário
129,2496,2024-11-30,2024-09-01,Involuntário


Inválidos após tratamento (esperado=0): 0
Formato final de 'desligamentos' após limpeza: (121, 4)


### Criação de variáveis - Feature Engineering

- Como mencionei no início, eu quis nessa etapa, já criar algumas variáveis complementares que possam auxiliar a Análise Exploratória ou mesmo a criação dos modelos

In [9]:
# ----------------------------
# Gerando variável de tempo de casa, em meses
# ----------------------------

hoje = pd.Timestamp.today().normalize()
tempo_meses = ((hoje - colab["data_admissao"]).dt.days // 30)
colab["tempo_casa_meses"] = tempo_meses.fillna(0).clip(lower=0).astype(int)

In [10]:
# ----------------------------
# Gerando variável de engajamento médio
# ----------------------------

eng_medio = (
    rh_mensal.groupby("colaborador_id")["engajamento"]
    .mean()
    .rename("engajamento_medio")
)

In [11]:
# ----------------------------
# Gerando variável de média de horas extras
# ----------------------------

hex_medio = (
    rh_mensal.groupby("colaborador_id")["horas_extras"]
    .mean()
    .rename("horas_extras_media")
)

In [12]:
# ----------------------------
# Gerando variável de performance média
# ----------------------------

perf_medio = (
    rh_mensal.groupby("colaborador_id")["performance"]
    .mean()
    .rename("performance_media")
)

In [13]:
# ----------------------------
# Merge com tabela de colaboradores
# ----------------------------

colab = colab.merge(eng_medio, on="colaborador_id", how="left") # engajamento médio
colab = colab.merge(hex_medio, on="colaborador_id", how="left") # média de horas extras
colab = colab.merge(perf_medio, on="colaborador_id", how="left") # performance média
colab.head(1)

,colaborador_id,area,nivel_cargo,genero,raca,data_admissao,idade,salario_base_inicial,tempo_casa_meses,engajamento_medio,horas_extras_media,performance_media
0,1,Tecnologia,Sênior,F,Parda,2018-09-25,37,11366.628604,90,65.452687,19.11853,3.030411


### Verificação de outliers

In [14]:
# ----------------------------
# Outliers
# ----------------------------

# Função para verificar a existência, mas nesse caso estou considerando o Intervalo Interquartil para calcular o limite inferior e o superior
# Desta forma, considerando que valores acima deste limite (nesta análise), seriam outliers

def detectar_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    return ((series < limite_inferior) | (series > limite_superior)).sum()

print("Outliers em horas extras:", detectar_outliers(colab["horas_extras_media"]))
print("Outliers em engajamento:", detectar_outliers(colab["engajamento_medio"]))
print("Outliers em performance:", detectar_outliers(colab["performance_media"]))

Outliers em horas extras: 363
Outliers em engajamento: 0
Outliers em performance: 7


In [15]:
# Já para o tratamento dos outliers, eu faço de forma diferente
# Aqui eu opto por truncar (impor limite) para valores extremos, de modo que fiquem como limites "aceitáveis", ou seja, os 1% maiores e 1% menores valores
# são truncados dentro dos percentis P1 (mínimo) e P99 (máximo), conforme a regra abaixo
'''
Se o valor é < P1 (1º percentil) → vira P1
Se o valor é > P99 (99º percentil) → vira P99
O resto permanece igual
'''

def truncar(serie, p=0.01):
    minimo = serie.quantile(p)
    maximo = serie.quantile(1-p)
    return serie.clip(minimo, maximo)

colab["horas_extras_media"] = truncar(colab["horas_extras_media"])
colab["engajamento_medio"] = truncar(colab["engajamento_medio"])
colab["performance_media"] = truncar(colab["performance_media"])


print("Outliers após tratamento:")
print("Horas extras:", detectar_outliers(colab["horas_extras_media"]))
print("Engajamento:", detectar_outliers(colab["engajamento_medio"]))
print("Performance:", detectar_outliers(colab["performance_media"]))

Outliers após tratamento:
Horas extras: 363
Engajamento: 0
Performance: 0


As horas extras continuam sendo classificadas como outliers porque, embora ultrapassem os limites definidos pelos limites superiores ou inferiores, elas ainda estão dentro da faixa entre os percentis P1 e P99 usada no truncamento proposto. 

Desta forma, apesar de serem valores altos, não são extremos segundo esse critério. Como esses picos refletem a dinâmica realista de sobrecarga no cenário simulado, decidi não aplicar uma limitação maior que essa para essa variável, visando atender critérios mais reais deste cenário.

### Relatório final de qualidade (últimos pontos de verificação)

- Verificar range dos indicadores de engajamento e performance
- Checar se os colaboradores nas bases tem alguma duplicidade
- Conferir se para um colaborador tem no máximo 1 desligamento
- Analisar se os colaboradores da tabela de desligamento estão presentes na tabela de colaboradores
- Fazer o mesmo do item acima, mas para a tabela de RH Mensal

In [16]:
# ----------------------------
# Teste final de qualidade
# ----------------------------

# Validando o range dos indicadores, para o intervalo proposto:
# Engajamento --- 0 a 100
# Performance --- 1 a 5

print(colab["engajamento_medio"].between(0,100).all())
print(colab["performance_media"].between(1,5).all())

True
True


In [17]:
# Confirmando a NÃO existência de duplicidade

print(
    colab.loc[
    colab.duplicated(subset=["colaborador_id", "data_admissao"], keep=False)
    ]
)

print(
    rh_mensal.loc[
    rh_mensal.duplicated(subset=["colaborador_id", "mes_ano"], keep=False)
    ]
)

print(
    promocoes.loc[
    promocoes.duplicated(subset=["colaborador_id", "data_promocao"], keep=False)
    ]
)

print(
    desligamentos.loc[
    desligamentos.duplicated(subset=["colaborador_id", "data_desligamento"], keep=False)
    ]
)

Empty DataFrame
Columns: [colaborador_id, area, nivel_cargo, genero, raca, data_admissao, idade, salario_base_inicial, tempo_casa_meses, engajamento_medio, horas_extras_media, performance_media]
Index: []
Empty DataFrame
Columns: [colaborador_id, mes_ano, engajamento, performance, horas_extras]
Index: []
Empty DataFrame
Columns: [colaborador_id, data_promocao, nivel_anterior, nivel_novo]
Index: []
Empty DataFrame
Columns: [colaborador_id, data_desligamento, tipo_desligamento, custo_reposicao]
Index: []


In [18]:
# Conferir se para um colaborador tem no máximo 1 desligamento

desligamentos["colaborador_id"].is_unique

True

In [19]:
# Analisar se os colaboradores da tabela de desligamento estão presentes na tabela de colaboradores

desligamentos.loc[
    ~desligamentos["colaborador_id"].isin(colab["colaborador_id"])
]

,colaborador_id,data_desligamento,tipo_desligamento,custo_reposicao


In [20]:
# Analisar se os colaboradores da tabela de RH estão presentes na tabela de colaboradores

rh_mensal.loc[
    ~rh_mensal["colaborador_id"].isin(colab["colaborador_id"])
]

,colaborador_id,mes_ano,engajamento,performance,horas_extras


**Resumo:** Após todas as análises, transformações e criações dos novos indicadores, entendo que temos uma tabela de base analítica, disponível para Análise Exploratória e construção de um modelo robusto. Com isso, seguirei com o salvamento desta base de dados.

In [21]:
# ----------------------------
# Informações gerais pós transformações e tratamentos
# ----------------------------

print("Informações - Tabela de Colaboradores\n")
colab.info()

print("\nInformações - Tabela de RH Mensal\n")
rh_mensal.info()

print("\nInformações - Tabela de Promoções\n")
promocoes.info()

print("\nInformações - Tabela de Desligamentos\n")
desligamentos.info()

Informações - Tabela de Colaboradores

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   colaborador_id        2500 non-null   int64         
 1   area                  2500 non-null   str           
 2   nivel_cargo           2500 non-null   str           
 3   genero                2500 non-null   str           
 4   raca                  2500 non-null   str           
 5   data_admissao         2500 non-null   datetime64[us]
 6   idade                 2500 non-null   int64         
 7   salario_base_inicial  2500 non-null   float64       
 8   tempo_casa_meses      2500 non-null   int64         
 9   engajamento_medio     2500 non-null   float64       
 10  horas_extras_media    2500 non-null   float64       
 11  performance_media     2500 non-null   float64       
dtypes: datetime64[us](1), float64(4), int64(3), str(

## Salvando Base de Dados Processada e ABT

### Bases de Dados - Outputs

In [22]:
# ----------------------------
# Salvando bases após processamentos
# ----------------------------

from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

colab.to_csv(output_dir / "colab_processed.csv", index=False)
rh_mensal.to_csv(output_dir / "rh_mensal_processed.csv", index=False)
promocoes.to_csv(output_dir / "promocoes_processed.csv", index=False)
desligamentos.to_csv(output_dir / "desligamentos_processed.csv", index=False)

### Montagem e salvamento da ABT
- Usarei a base de rh_mensal, pois a análise visa extrair um período curto de 24 meses e não o histórico completo da base de colaboradores

In [23]:
# ----------------------------
# Gerando ABT para uso analítico
# ----------------------------

abt = rh_mensal.copy()

In [24]:
abt = abt.merge(
    colab,
    on="colaborador_id",
    how="left",
    validate="m:1"
)

In [25]:
abt = abt.merge(
    desligamentos,
    on="colaborador_id",
    how="left",
    validate="m:1"
)

In [26]:
# criei uma target de desligamento com 1 mês de antecedência ao desligamento

abt["target_desligamento"] = (
    abt["data_desligamento"].dt.to_period("M")
    == (abt["mes_ano"] + pd.DateOffset(months=1)).dt.to_period("M")
).astype(int)

In [27]:
abt["ativo_no_mes"] = (
    abt["data_desligamento"].isna()
) | (abt["mes_ano"] <= abt["data_desligamento"])

In [28]:
abt.shape

(60000, 21)

In [29]:
primeira_promocao = (
    promocoes
    .sort_values("data_promocao")
    .drop_duplicates("colaborador_id")
)
primeira_promocao.head()

,colaborador_id,data_promocao,nivel_anterior,nivel_novo
180,1213,2024-01-01,Pleno,Sênior
23,122,2024-01-01,Sênior,Líder
145,941,2024-01-01,Júnior,Pleno
14,69,2024-01-01,Sênior,Líder
142,885,2024-01-01,Júnior,Pleno


In [30]:
abt = abt.merge(
    primeira_promocao[["colaborador_id", "data_promocao"]],
    on="colaborador_id",
    how="left"
)

In [31]:
abt["foi_promovido"] = (
    abt["data_promocao"] <= abt["mes_ano"]
).astype(int)

In [32]:
# Verificando se não há nulos nem no id do colaborador e nem no mês e ano

assert abt["colaborador_id"].notna().all()
assert abt["mes_ano"].notna().all()
assert len(abt) <= len(rh_mensal)

In [33]:
# ----------------------------
# Informações da estrutura final da ABT
# ----------------------------

abt.head(1)
print(abt.info())

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   colaborador_id        60000 non-null  int64         
 1   mes_ano               60000 non-null  datetime64[us]
 2   engajamento           60000 non-null  float64       
 3   performance           60000 non-null  float64       
 4   horas_extras          60000 non-null  float64       
 5   area                  60000 non-null  str           
 6   nivel_cargo           60000 non-null  str           
 7   genero                60000 non-null  str           
 8   raca                  60000 non-null  str           
 9   data_admissao         60000 non-null  datetime64[us]
 10  idade                 60000 non-null  int64         
 11  salario_base_inicial  60000 non-null  float64       
 12  tempo_casa_meses      60000 non-null  int64         
 13  engajamento_medio     60000

In [34]:
abt.to_csv(output_dir / "abt_turnover.csv", index=False)

In [35]:
end = time.perf_counter()
print(f"Tempo total: {end - start:.2f} segundos")

Tempo total: 3.56 segundos
